In [10]:
from sklearn.model_selection import train_test_split
import pandas as pd

# Load the data
df = pd.read_csv('edited2_measures.csv', delimiter=';')
# Convert string numbers with commas to floats
for column in df.columns:
    if df[column].dtype == 'object':
        try:
            df[column] = df[column].str.replace(',', '.').astype(float)
        except ValueError:
            pass  # If conversion fails, it's probably a categorical column
X = df.drop(columns=['activity'])
y = df['activity']
# Split the data 0.7 train & 0.3 test
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)

#############################################################################

from sklearn.metrics import accuracy_score
from xgboost import XGBClassifier
from sklearn.preprocessing import LabelEncoder

# Convert categorical variables in X to numerical variables using one-hot encoding
L = pd.get_dummies(X)
# Encode the target labels
label_encoder = LabelEncoder()
Y_encoded = label_encoder.fit_transform(y)
# Split the data into training and testing sets
L_train, L_test, n_train, n_test = train_test_split(L, Y_encoded, test_size=0.2, random_state=42)

# Set random seef for reproducibility
xgb = XGBClassifier(random_state=42, use_label_encoder=False, eval_metric='logloss')
xgb.fit(L_train, n_train)
y_pred_xgb = xgb.predict(L_test)
accuracy_xgb = accuracy_score(n_test, y_pred_xgb)
print(f'XGBoost Accuracy: {accuracy_xgb:.4f}')


###############################################################################


# predict classes of edited_to_predict.csv with unkown classes of activity
# using the stacking model
df = pd.read_csv('edited2_to_predict.csv', delimiter=';')
# Convert string numbers with commas to floats
for column in df.columns:
    if df[column].dtype == 'object':
        try:
            df[column] = df[column].str.replace(',', '.').astype(float)
        except ValueError:
            pass  # If conversion fails, it's probably a categorical column
# edited2_to_predict.csv has no activity column
# so we have to predict it
X = df
y_pred = xgb.predict(X)
# Convert numerical labels back to string labels
y_pred = label_encoder.inverse_transform(y_pred)

# save the output in new csv file
df['activity'] = y_pred
df.to_csv('xgboost_predictions.csv', sep=';', index=False)
print('Predicted classes saved in xgboost_predictions.csv')


XGBoost Accuracy: 0.9946
Predicted classes saved in xgboost_predictions.csv
